In [7]:
"""
Script to compare two CSV files using fuzzy matching
- Compares ai_screening_full_results_clean.csv (ML predictions) with review_647084_included_csv_20260123224523.csv (gold truth)
- Uses fuzzy matching on titles to identify the same studies
- Calculates sensitivity and specificity metrics
- Outputs a detailed comparison report
"""

import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from datetime import datetime
import os

# File paths
ML_FILE = "/Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/ai_screening_include.csv"
GOLD_FILE = "/Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/human_screening/review_647084_included_csv_20260123224523.csv"
OUTPUT_DIR = "/Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs"

def load_data():
    """Load both CSV files"""
    print("Loading data files...")
    ml_df = pd.read_csv(ML_FILE)
    gold_df = pd.read_csv(GOLD_FILE)
    
    print(f"ML predictions file: {len(ml_df)} records")
    print(f"Gold truth file: {len(gold_df)} records")
    
    return ml_df, gold_df

def normalize_title(title):
    """Normalize title for better matching"""
    if pd.isna(title):
        return ""
    # Convert to lowercase, remove extra whitespace
    title = str(title).lower().strip()
    # Remove special characters
    title = ''.join(c if c.isalnum() or c.isspace() else ' ' for c in title)
    # Remove extra spaces
    title = ' '.join(title.split())
    return title

def fuzzy_match_titles(ml_df, gold_df, threshold=95):
    """
    Match titles between ML predictions and gold truth using fuzzy matching
    Returns a mapping dictionary and match details
    """
    print(f"\nPerforming fuzzy matching (threshold: {threshold})...")
    
    # Normalize titles
    ml_df['normalized_title'] = ml_df['Title'].apply(normalize_title)
    gold_df['normalized_title'] = gold_df['Title'].apply(normalize_title)
    
    matches = []
    ml_to_gold_map = {}
    
    for ml_idx, ml_row in ml_df.iterrows():
        ml_title = ml_row['normalized_title']
        if not ml_title:
            continue
            
        best_match = None
        best_score = 0
        
        for gold_idx, gold_row in gold_df.iterrows():
            gold_title = gold_row['normalized_title']
            if not gold_title:
                continue
            
            # Use token sort ratio for better matching with word order variations
            score = fuzz.token_sort_ratio(ml_title, gold_title)
            
            if score > best_score:
                best_score = score
                best_match = gold_idx
        
        if best_score >= threshold:
            matches.append({
                'ml_index': ml_idx,
                'gold_index': best_match,
                'ml_title': ml_row['Title'],
                'gold_title': gold_df.loc[best_match, 'Title'],
                'match_score': best_score,
                'ml_pmid': ml_row.get('PMID', 'N/A'),
                'gold_authors': gold_df.loc[best_match, 'Authors']
            })
            ml_to_gold_map[ml_idx] = best_match
    
    print(f"Found {len(matches)} matches above threshold {threshold}")
    
    return matches, ml_to_gold_map

def calculate_metrics(ml_df, gold_df, matches, ml_to_gold_map):
    """
    Calculate sensitivity and specificity metrics
    Treating gold_df as the ground truth (all are included studies)
    """
    print("\nCalculating performance metrics...")
    
    # All studies in gold truth are "included" (true positives in reality)
    gold_included = set(gold_df.index)
    
    # Studies in ML that matched with gold truth
    ml_matched_indices = set(ml_to_gold_map.keys())
    
    # Studies in ML that did NOT match with gold truth
    ml_unmatched_indices = set(ml_df.index) - ml_matched_indices
    
    # True Positives: Studies correctly identified (in both ML and gold)
    true_positives = len(matches)
    
    # False Negatives: Studies in gold truth but NOT in ML predictions
    gold_matched_indices = set(ml_to_gold_map.values())
    false_negatives = len(gold_included - gold_matched_indices)
    
    # False Positives: Studies in ML but NOT in gold truth
    false_positives = len(ml_unmatched_indices)
    
    # True Negatives: Cannot be calculated as we don't have the full universe of excluded studies
    # We only have included studies in both datasets
    
    # Calculate metrics
    sensitivity = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    
    # Precision (Positive Predictive Value)
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    
    # F1 Score
    f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    
    metrics = {
        'true_positives': true_positives,
        'false_negatives': false_negatives,
        'false_positives': false_positives,
        'sensitivity_recall': sensitivity,
        'precision': precision,
        'f1_score': f1_score,
        'total_gold_studies': len(gold_df),
        'total_ml_studies': len(ml_df),
        'match_rate': true_positives / len(gold_df) if len(gold_df) > 0 else 0
    }
    
    return metrics

def generate_report(ml_df, gold_df, matches, metrics, output_dir):
    """Generate comprehensive comparison report"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = os.path.join(output_dir, f"comparison_report_{timestamp}.txt")
    matches_file = os.path.join(output_dir, f"matched_studies_{timestamp}.csv")
    in_ml_not_gold_file = os.path.join(output_dir, f"in_ml_not_in_gold_{timestamp}.csv")
    in_gold_not_ml_file = os.path.join(output_dir, f"in_gold_not_in_ml_{timestamp}.csv")
    
    # Create matches dataframe
    if matches:
        matches_df = pd.DataFrame(matches)
        matches_df.to_csv(matches_file, index=False)
        print(f"\nMatched studies saved to: {matches_file}")
    
    # Studies in ML but not in gold
    ml_matched_indices = set([m['ml_index'] for m in matches])
    in_ml_not_gold = ml_df[~ml_df.index.isin(ml_matched_indices)][['PMID', 'Title', 'Authors', 'Date', 'Journal']]
    in_ml_not_gold.to_csv(in_ml_not_gold_file, index=False)
    print(f"Studies in ML but not in gold truth saved to: {in_ml_not_gold_file}")
    
    # Studies in gold but not in ML
    gold_matched_indices = set([m['gold_index'] for m in matches])
    in_gold_not_ml = gold_df[~gold_df.index.isin(gold_matched_indices)][['Title', 'Authors', 'Published Year', 'Journal']]
    in_gold_not_ml.to_csv(in_gold_not_ml_file, index=False)
    print(f"Studies in gold truth but not in ML saved to: {in_gold_not_ml_file}")
    
    # Generate text report
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("COMPARISON REPORT: ML Predictions vs Gold Truth (Human Screening)\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"ML Predictions File: {os.path.basename(ML_FILE)}\n")
        f.write(f"Gold Truth File: {os.path.basename(GOLD_FILE)}\n\n")
        
        f.write("-" * 80 + "\n")
        f.write("DATASET SUMMARY\n")
        f.write("-" * 80 + "\n")
        f.write(f"Total studies in ML predictions: {metrics['total_ml_studies']}\n")
        f.write(f"Total studies in gold truth: {metrics['total_gold_studies']}\n\n")
        
        f.write("-" * 80 + "\n")
        f.write("MATCHING RESULTS\n")
        f.write("-" * 80 + "\n")
        f.write(f"Studies found in both datasets: {metrics['true_positives']}\n")
        f.write(f"Studies in ML but NOT in gold truth: {metrics['false_positives']}\n")
        f.write(f"Studies in gold truth but NOT in ML: {metrics['false_negatives']}\n\n")
        
        f.write("-" * 80 + "\n")
        f.write("PERFORMANCE METRICS\n")
        f.write("-" * 80 + "\n")
        f.write(f"Sensitivity (Recall): {metrics['sensitivity_recall']:.2%}\n")
        f.write(f"  - How many of the gold truth studies were captured by ML\n")
        f.write(f"  - Formula: TP / (TP + FN) = {metrics['true_positives']} / ({metrics['true_positives']} + {metrics['false_negatives']})\n\n")
        
        f.write(f"Precision (Positive Predictive Value): {metrics['precision']:.2%}\n")
        f.write(f"  - How many ML predictions were correct (matched gold truth)\n")
        f.write(f"  - Formula: TP / (TP + FP) = {metrics['true_positives']} / ({metrics['true_positives']} + {metrics['false_positives']})\n\n")
        
        f.write(f"F1 Score: {metrics['f1_score']:.2%}\n")
        f.write(f"  - Harmonic mean of precision and recall\n")
        f.write(f"  - Formula: 2 * (Precision * Recall) / (Precision + Recall)\n\n")
        
        f.write(f"Match Rate: {metrics['match_rate']:.2%}\n")
        f.write(f"  - Percentage of gold truth studies found in ML\n\n")
        
        f.write("-" * 80 + "\n")
        f.write("INTERPRETATION\n")
        f.write("-" * 80 + "\n")
        f.write("Note: Specificity cannot be calculated because we don't have the full\n")
        f.write("universe of excluded studies (true negatives). Both datasets contain\n")
        f.write("only included studies.\n\n")
        
        f.write("Sensitivity/Recall indicates how well the ML approach identified the\n")
        f.write("relevant studies that were manually screened and included.\n\n")
        
        f.write("Precision indicates what proportion of ML-predicted studies actually\n")
        f.write("appear in the manually curated gold truth set.\n\n")
        
        f.write("-" * 80 + "\n")
        f.write("OUTPUT FILES GENERATED\n")
        f.write("-" * 80 + "\n")
        f.write(f"1. This report: {os.path.basename(report_file)}\n")
        f.write(f"2. Matched studies: {os.path.basename(matches_file)}\n")
        f.write(f"3. In ML but not gold: {os.path.basename(in_ml_not_gold_file)}\n")
        f.write(f"4. In gold but not ML: {os.path.basename(in_gold_not_ml_file)}\n")
        f.write("\n" + "=" * 80 + "\n")
    
    print(f"\nComparison report saved to: {report_file}")
    
    # Print summary to console
    print("\n" + "=" * 80)
    print("PERFORMANCE SUMMARY")
    print("=" * 80)
    print(f"Sensitivity (Recall): {metrics['sensitivity_recall']:.2%}")
    print(f"Precision: {metrics['precision']:.2%}")
    print(f"F1 Score: {metrics['f1_score']:.2%}")
    print(f"Match Rate: {metrics['match_rate']:.2%}")
    print("\nTrue Positives (in both): {0}".format(metrics['true_positives']))
    print(f"False Positives (in ML only): {metrics['false_positives']}")
    print(f"False Negatives (in gold only): {metrics['false_negatives']}")
    print("=" * 80)

def main():
    """Main execution function"""
    print("\n" + "=" * 80)
    print("CSV COMPARISON WITH FUZZY MATCHING")
    print("=" * 80)
    
    # Load data
    ml_df, gold_df = load_data()
    
    # Perform fuzzy matching
    matches, ml_to_gold_map = fuzzy_match_titles(ml_df, gold_df, threshold=85)
    
    # Calculate metrics
    metrics = calculate_metrics(ml_df, gold_df, matches, ml_to_gold_map)
    
    # Generate report
    generate_report(ml_df, gold_df, matches, metrics, OUTPUT_DIR)
    
    print("\n✓ Comparison complete!")

if __name__ == "__main__":
    main()



CSV COMPARISON WITH FUZZY MATCHING
Loading data files...
ML predictions file: 52 records
Gold truth file: 36 records

Performing fuzzy matching (threshold: 85)...
Found 34 matches above threshold 85

Calculating performance metrics...

Matched studies saved to: /Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/matched_studies_20260123_162232.csv
Studies in ML but not in gold truth saved to: /Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/in_ml_not_in_gold_20260123_162232.csv
Studies in gold truth but not in ML saved to: /Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/in_gold_not_in_ml_20260123_162232.csv

Comparison report saved to: /Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/comparison_report_20260123_162232.txt

PERFORMANCE SUMMARY
Sensitivity (Recall): 85.00%
Precision: 65.38%
F1 Score: 73.91%
Match Rate: 94.44%

True Positives (in both): 34


In [4]:
#create a csv with just ai_decision = include or uncertain
import pandas as pd
import os
df = pd.read_csv("/Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/ai_screening_full_results.csv")
df_clean = df[df['ai_decision'].astype(str).str.strip().str.lower().
isin(["include", "uncertain"])]
df_clean.to_csv("/Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/ai_screening_include.csv", index=False)
print("✓ Cleaned AI screening results saved.")

✓ Cleaned AI screening results saved.
